# Environment

## Imports

In [1]:
%load_ext autoreload

In [2]:
# Reload only modules imported with %aimport
%autoreload 1

# Mark these modules for auto-reload
%aimport fetch_series.core

In [3]:
import httpx
from httpx import RemoteProtocolError, ConnectError, ReadError
from httpx_retries import Retry, RetryTransport
import pandas as pd
import asyncio
import json
import re
import time
from tqdm.auto import tqdm
from irods.session import iRODSSession
import fetch_series
from fetch_series.core import *
from typing import List, Tuple

## Global variables

In [4]:
irods_base = '/archive/cellgeni/datasets'
env_file = "/nfs/users/nfs_c/cellgeni-su/.irods/irods_environment.json"

## Helper functions

In [5]:
def geo2bioproject(series: str) -> str:
    """Convert a GEO series ID to a BioProject ID."""
    # Search GSE in GEO
    geo_id = geo_dataset_id(series=series, stype="gse")

    # Get GSE info
    dataset = fetch_series.eutils_summary(
        ids=geo_id,
        db="gds",
    )
    if "result" not in dataset or geo_id not in dataset["result"]:
        raise ValueError(f"Failed to retrieve summary for GEO ID {geo_id}")
    return dataset["result"][geo_id]["bioproject"]

In [6]:
series = "E-MTAB-11264"
base = f"https://www.ebi.ac.uk/biostudies/files/{series}/{series}.idf.txt"
with httpx.Client(transport=transport) as client:
    response = client.get(base, follow_redirects=True, timeout=30)
response.raise_for_status()

<Response [200 OK]>

In [7]:
pattern = re.compile(r"\[SecondaryAccession\]\s+((?:ERP|EGAD)\d+)")
matches = re.findall(pattern, response.text)
print(matches)

['ERP135004']


In [8]:
ena2bioproject(series="ERP126408")

['PRJEB42537']

In [9]:
ena2ae(series="ERP126408")

['E-MTAB-10018']

In [10]:
bioproject2ena(series="PRJEB42537")

'ERP126408'

In [11]:
base = "https://www.ebi.ac.uk/ena/portal/api/filereport"
params = {
    "accession": "ERP126408",
    "result": "read_run",
    "format": "json",
    "fields": "study_alias,study_accession,",
}

response = httpx.get(base, params=params, timeout=10.0, follow_redirects=True)
print(json.dumps(response.json(), indent=2))

[
  {
    "run_accession": "ERR5104752",
    "study_alias": "E-MTAB-10018",
    "study_accession": "PRJEB42537"
  },
  {
    "run_accession": "ERR5104753",
    "study_alias": "E-MTAB-10018",
    "study_accession": "PRJEB42537"
  },
  {
    "run_accession": "ERR5104755",
    "study_alias": "E-MTAB-10018",
    "study_accession": "PRJEB42537"
  },
  {
    "run_accession": "ERR5104759",
    "study_alias": "E-MTAB-10018",
    "study_accession": "PRJEB42537"
  },
  {
    "run_accession": "ERR5104760",
    "study_alias": "E-MTAB-10018",
    "study_accession": "PRJEB42537"
  },
  {
    "run_accession": "ERR5104762",
    "study_alias": "E-MTAB-10018",
    "study_accession": "PRJEB42537"
  },
  {
    "run_accession": "ERR5104765",
    "study_alias": "E-MTAB-10018",
    "study_accession": "PRJEB42537"
  },
  {
    "run_accession": "ERR5104768",
    "study_alias": "E-MTAB-10018",
    "study_accession": "PRJEB42537"
  },
  {
    "run_accession": "ERR5104770",
    "study_alias": "E-MTAB-10018",
    

# Get a list of all processed series

In [12]:
filter_cond = lambda coll: coll.name.startswith("GSE") or coll.name.startswith("E-MTAB") or coll.name.startswith("PRJ")
with iRODSSession(irods_env_file=env_file) as session:
    collection = session.collections.get(irods_base)
    processed_series = [coll.name for coll in collection.subcollections if filter_cond(coll)]

print(f"Found {len(processed_series)} processed series")
print(f"ArrayExpress series: {len(list(filter(lambda x: "E-MTAB" in x, processed_series)))}")
print(f"Gene Expression Omnibus series: {len(list(filter(lambda x: "GSE" in x, processed_series)))}")
print(f"Bioproject series: {len(list(filter(lambda x: "PRJ" in x, processed_series)))}")

Found 2884 processed series
ArrayExpress series: 117
Gene Expression Omnibus series: 2757
Bioproject series: 10


# Array Express

## Get a list of all AE series with RNA sequencing

Function to get all ArrayExpress accessions

In [13]:
BASE = "https://www.ebi.ac.uk/biostudies/api/v1/arrayexpress/search"
EMTAB = re.compile(r"^E-MTAB-\d+$")

async def accessions_from_page(page, pagesize, client):
    params = {
        "query": "",  # broad match, we regex-filter below
        "pageSize": min(pagesize, 100),  # 100 is commonly used
        "page": page,
        "facet.technology": "sequencing assay",
        "facet.assay_by_molecule": "rna assay",
    }
    r = await client.get(BASE, params=params)
    r.raise_for_status()
    data = r.json()

    hits = data.get("hits") or []
    acc_list = []

    for h in hits:
        acc = h.get("accno") or h.get(
            "accession"
        )  # accno is what BioStudies uses in many responses
        if acc and EMTAB.match(acc):
            acc_list.append(acc)
    return acc_list

Get a total number of accessions

In [14]:
base = "https://www.ebi.ac.uk/biostudies/api/v1/arrayexpress/search"
emtab = re.compile(r"^E-MTAB-\d+$")

r = httpx.get(
    base,
    params={
        "query": "",
        "pageSize": 1,
        "page": 1,
        "facet.technology": "sequencing assay",
        "facet.assay_by_molecule": "rna assay",
    },
)
r.raise_for_status()
total_hits = r.json().get("totalHits", 0)
print(total_hits)

ReadTimeout: The read operation timed out

In [ ]:
r.json()

{'page': 1,
 'pageSize': 1,
 'totalHits': 15896,
 'isTotalHitsExact': True,
 'sortBy': 'release_date',
 'sortOrder': 'descending',
 'hits': [{'accession': 'E-MTAB-15733',
   'type': 'study',
   'title': 'RNA-seq experiment of V79 cells treated with UV light and incubated either in monoculture or coculture with untreated cells',
   'author': 'Jelena Budimir',
   'links': 1,
   'files': 3,
   'release_date': '2026-03-09',
   'views': 0,
   'isPublic': True}],
 'query': None,
 'facets': {'facet.technology': ['sequencing assay'],
  'facet.assay_by_molecule': ['rna assay']}}

Get all accessions

In [ ]:
concurrency = 20
sem = asyncio.Semaphore(concurrency)


limits = httpx.Limits(
    max_connections=concurrency,
    max_keepalive_connections=concurrency,
)

retry = Retry(
    total=5,
    backoff_factor=0.1,  # Wait 0.3s, 0.6s, 1.2s, 2.4s, 4.8s between retries
    status_forcelist=[
        408,  # Request Timeout - server took too long to respond
        429,  # Too Many Requests - rate limiting (common with NCBI)
        500,  # Internal Server Error - temporary server issue
        502,  # Bad Gateway - gateway/proxy error
        503,  # Service Unavailable - server overloaded/maintenance
        504,  # Gateway Timeout - gateway didn't get response in time
    ],
)

transport = RetryTransport(retry=retry)
pagesize = 100
n_steps = total_hits // pagesize + 1
results = list()

async with httpx.AsyncClient(transport=transport, timeout=30.0, limits=limits) as client:

    async def bounded(page):
        async with sem:
            return await accessions_from_page(page, pagesize, client)

    tasks = [bounded(i) for i in range(1, n_steps + 1)]
    with tqdm(total=n_steps) as pbar:
        for task in asyncio.as_completed(tasks):
            results += await task
            pbar.update(1)
            pbar.refresh()

print(f"Found {len(results)} accessions")

  0%|          | 0/159 [00:00<?, ?it/s]

Found 7179 accessions


In [ ]:
sorted_accessions = sorted(set(results), key=lambda x: int(x.split("-")[-1]))

Save all AE accessions to a file

In [ ]:
with open(f"data/emtab_accessions_{time.strftime('%Y-%m-%d')}.txt", "w") as f:
    for acc in sorted_accessions:
        f.write(f"{acc}\n")

## Get secondary accessions for all AE series

Get secondary accessions for all AE series

In [ ]:
accession = "E-MTAB-6126"
accession = "E-MTAB-10018"

In [ ]:
async def read_enaruns_async(
    series: str,
    format: Literal["json", "tsv"] = "json",
    fields: str = "study_accession,experiment_accession,run_accession",
    limit: int | None = None,
    client: httpx.AsyncClient | None = None
) -> List[Dict[str, Any]] | None:
    """
    Retrieves ENA run accessions associated with a given series from the EBI ENA database
    Args:
        series (str): series identifier to retrieve ENA run accessions for (e.g., E-MTAB-10018)
        format (Literal['json', 'tsv']): format to return the results in (default: 'json')
        fields (str): comma-separated list of fields to include in the results (default: 'study_accession,experiment_accession,run_accession')

    Returns:
        List[Dict[str, Any]] | None: A list of dictionaries containing ENA run information, or None if no runs are found
    """
    base = "https://www.ebi.ac.uk/ena/portal/api/filereport"
    params = {
        "accession": series,
        "result": "read_run",
        "format": format,
        "fields": fields,
        "limit": limit,
    }
    response = await client.get(base, params=params, timeout=10.0, follow_redirects=True)
    #response.raise_for_status()

    if format == "json":
        return response.json()
    elif format == "tsv":
        reader = csv.DictReader(io.StringIO(response.text), delimiter="\t")
        return list(reader)


async def ena2bioproject_async(series: str, client: httpx.AsyncClient) -> List[str]:
    """
    Retrieves the BioProject ID associated with a given ENA series from the EBI ENA database
    Args:
        series (str): series identifier to retrieve the BioProject ID for (e.g., E-MTAB-10018)
        client (httpx.AsyncClient): async HTTP client to use for requests

    Returns:
        List[str]: A list of BioProject IDs associated with the series, or an empty list if no BioProjects are found
    """
    runs = await read_enaruns_async(series=series, format="json", fields="study_accession", client=client)
    try:
        if not runs or len(runs) == 0 or "study_accession" not in runs[0]:
            #logging.warning("No BioProject found for '%s'", series)
            return []
    except Exception as e:
        print(f"Error processing runs for series '{series}': {e}")
        print(f"Runs data: {runs}")
        raise e
    if not runs or len(runs) == 0 or "study_accession" not in runs[0]:
        #logging.warning("No BioProject found for %s", series)
        return []
    accessions = set(run["study_accession"] for run in runs if "study_accession" in run)
    return list(accessions)


async def ae2biosamples_async(series: str, client: httpx.AsyncClient) -> List[str]:
    """
    Retrieves biosample IDs associated with a given series from the EBI BioStudies database
    Args:
        series (str): series identifier to retrieve biosample IDs for (e.g., E-MTAB-10018)
        client (httpx.AsyncClient): async HTTP client to use for requests

    Returns:
        List[str] | None: A list of biosample IDs associated with the series, or None if no biosamples are found
    """
    base = f"https://www.ebi.ac.uk/biostudies/files/{series}/{series}.sdrf.txt"
    response = await client.get(base, follow_redirects=True, timeout=30)
    # response.raise_for_status()

    biosamples = []
    reader = csv.DictReader(io.StringIO(response.text), delimiter="\t")
    biosamples = [
        row.get("Comment[BioSD_SAMPLE]").strip()
        for row in reader
        if "Comment[BioSD_SAMPLE]" in row
    ]
    biosamples = list(filter(None, biosamples))  # Remove empty strings
    return biosamples


async def ae2secondary_async(series: str, client: httpx.AsyncClient) -> Tuple[str, List[str]]:
    """
    Async version: Retrieves SecondaryAccession values associated with a given series from the EBI BioStudies database
    Args:
        series (str): series identifier to retrieve SecondaryAccession values for (e.g., E-MTAB-10018)
        client (httpx.AsyncClient): async HTTP client to use for requests

    Returns:
        Tuple[str, List[str]]: A tuple containing the series identifier and a list of SecondaryAccession values associated with the series
    """
    try:
        base = f"https://www.ebi.ac.uk/biostudies/files/{series}/{series}.idf.txt"
        response = await client.get(base, follow_redirects=True, timeout=30)
        #response.raise_for_status()

        # Check if there are SecondaryAccession values in the idf file
        pattern = re.compile(r"Comment\s*\[SecondaryAccession\]\s*((?:ERP|EGA\w)\d+)")
        secondary = re.findall(pattern, response.text)

        # Try to get secondary accesssions from ENA if not found in idf file
        if not secondary:
            biosamples = await ae2biosamples_async(series=series, client=client)
            if not biosamples:
                # logging.warning(
                #     "No biosamples found for %s, cannot retrieve secondary accessions",
                #     series,
                # )
                return series, []

            # Attempt to retrieve secondary accessions from ENA for each biosample
            for biosample in biosamples:
                secondary_accessions = await ena2bioproject_async(series=biosample, client=client)
                if secondary_accessions:
                    secondary.extend(secondary_accessions)

            secondary = list(set(secondary))  # Remove duplicates

        if not secondary:
            #logging.warning("No secondary accessions found for %s", series)
            pass
        return series, secondary
    except Exception as e:
        print(f"Error processing series '{series}': {e}")
        raise e


In [ ]:
concurrency = 7
sem = asyncio.Semaphore(concurrency)


limits = httpx.Limits(
    max_connections=concurrency,
    max_keepalive_connections=0,
)

retry = Retry(
    total=5,
    backoff_factor=1,  # Wait 0.3s, 0.6s, 1.2s, 2.4s, 4.8s between retries
    status_forcelist=[
        400,  # Bad Request - malformed request (e.g., invalid series ID)
        403,  # Forbidden - access issues (e.g., private datasets)
        408,  # Request Timeout - server took too long to respond
        429,  # Too Many Requests - rate limiting (common with NCBI)
        500,  # Internal Server Error - temporary server issue
        502,  # Bad Gateway - gateway/proxy error
        503,  # Service Unavailable - server overloaded/maintenance
        504,  # Gateway Timeout - gateway didn't get response in time
    ],
    retry_on_exceptions=[RemoteProtocolError, ConnectError, ReadError],
)

transport = RetryTransport(retry=retry)
results = list()

async with httpx.AsyncClient(
    transport=transport, timeout=30.0, limits=limits
) as client:

    async def bounded(series):
        async with sem:
            return await ae2secondary_async(series, client)

    tasks = [bounded(series) for series in sorted_accessions if series != "E-MTAB-5448"]
    with tqdm(total=len(sorted_accessions)) as pbar:
        for task in asyncio.as_completed(tasks):
            results.append(await task)
            pbar.update(1)
            pbar.refresh()

  0%|          | 0/7179 [00:00<?, ?it/s]

ReadTimeout: 

There is some problem with E-MTAB-5448

Parse the results

In [22]:
parsed_results = list()
for series, secondary_list in results:
    erps = list()
    egas = list()
    others = list()
    for acc in secondary_list:
        if acc.startswith("ERP"):
            erps.append(acc)
        elif acc.startswith("EGA"):
            egas.append(acc)
        else:
            others.append(acc)
    parsed_results.append({
        "series": series,
        "erps": ",".join(erps),
        "egas": ",".join(egas),
        "others": ",".join(others),
        "n_erps": len(erps),
        "n_egas": len(egas),
        "n_others": len(others),
        "total": len(secondary_list),
        })

Create a DataFrame with the results

In [23]:
results_df = pd.DataFrame(parsed_results).sort_values(
    "series", key=lambda x: x.str.split("-").str[-1]
.astype(int))
results_df.head()

,series,erps,egas,others,n_erps,n_egas,n_others,total
4098,E-MTAB-71,,,,0,0,0,0
4095,E-MTAB-75,ERP001207,,,1,0,0,1
5724,E-MTAB-115,,,,0,0,0,0
5561,E-MTAB-284,ERP000211,,,1,0,0,1
5699,E-MTAB-305,ERP000257,,,1,0,0,1


In [24]:
results_df.sum(axis=0, numeric_only=True).to_frame(name="count").T

,n_erps,n_egas,n_others,total
count,6630,20,82,6732


In [25]:
results_df.total.value_counts().to_frame(name="count").T

total,1,0,2,4,3
count,6715,457,5,1,1


In [26]:
results_df[results_df["total"] > 1]

,series,erps,egas,others,n_erps,n_egas,n_others,total
7178,E-MTAB-2770,,,"PRJNA523380,PRJNA1262534",0,0,2,2
5213,E-MTAB-3358,,,"PRJDB5572,PRJDB1099,PRJDB2675,PRJDB2676",0,0,4,4
5360,E-MTAB-3578,,,"PRJDB1100,PRJDB1099,PRJDB4060",0,0,3,3
416,E-MTAB-3579,,,"PRJDB1100,PRJDB1099",0,0,2,2
2206,E-MTAB-4101,"ERP023737,ERP010353",,,2,0,0,2
3794,E-MTAB-4748,ERP007111,EGAS00001000593,,1,1,0,2
4666,E-MTAB-8818,"ERP120195,ERP104782",,,2,0,0,2


In [30]:
results_df.to_csv(f"data/emtab_accessions_with_secondary_{time.strftime('%Y-%m-%d')}.csv", index=False)

## Check that I can retrieve all secondary accessions for the list of processed AE series

In [ ]:
processed_ae = list(filter(lambda x: "E-MTAB" in x, processed_series))

In [ ]:
concurrency = 10
sem = asyncio.Semaphore(concurrency)


limits = httpx.Limits(
    max_connections=concurrency,
    max_keepalive_connections=0,  # Disable keepalive to avoid stale connection reuse
)

retry = Retry(
    total=5,
    backoff_factor=0.1,  # Wait 0.3s, 0.6s, 1.2s, 2.4s, 4.8s between retries
    status_forcelist=[
        408,  # Request Timeout - server took too long to respond
        429,  # Too Many Requests - rate limiting (common with NCBI)
        500,  # Internal Server Error - temporary server issue
        502,  # Bad Gateway - gateway/proxy error
        503,  # Service Unavailable - server overloaded/maintenance
        504,  # Gateway Timeout - gateway didn't get response in time
    ],
    retry_on_exceptions=[RemoteProtocolError, ConnectError, ReadError],
)

transport = RetryTransport(retry=retry)
results = list()

async with httpx.AsyncClient(
    transport=transport, timeout=30.0, limits=limits
) as client:

    async def bounded(series):
        async with sem:
            for attempt in range(5):
                try:
                    return await ae2secondary_async(series, client)
                except (RemoteProtocolError, ConnectError, ReadError) as e:
                    if attempt == 4:
                        raise
                    await asyncio.sleep(0.1 * (2 ** attempt))

    tasks = [bounded(series) for series in processed_ae]
    with tqdm(total=len(processed_ae)) as pbar:
        for task in asyncio.as_completed(tasks):
            results.append(await task)
            pbar.update(1)
            pbar.refresh()


  0%|          | 0/117 [00:00<?, ?it/s]

Parse the results

In [ ]:
parsed_results = list()
for series, secondary_list in results:
    erps = list()
    egas = list()
    others = list()
    for acc in secondary_list:
        if acc.startswith("ERP"):
            erps.append(acc)
        elif acc.startswith("EGA"):
            egas.append(acc)
        else:
            others.append(acc)
    parsed_results.append({
        "series": series,
        "erps": ",".join(erps),
        "egas": ",".join(egas),
        "others": ",".join(others),
        "n_erps": len(erps),
        "n_egas": len(egas),
        "n_others": len(others),
        "total": len(secondary_list),
        })

Create a DataFrame with the results

In [ ]:
results_df = pd.DataFrame(parsed_results).sort_values(
    "series", key=lambda x: x.str.split("-").str[-1]
.astype(int))
results_df.head()

,series,erps,egas,others,n_erps,n_egas,n_others,total
33,E-MTAB-3929,ERP012552,,,1,0,0,1
36,E-MTAB-6108,ERP108448,,,1,0,0,1
44,E-MTAB-6505,,,,0,0,0,0
42,E-MTAB-6524,ERP115108,,,1,0,0,1
52,E-MTAB-6687,ERP110437,,,1,0,0,1


In [ ]:
results_df.sum(axis=0, numeric_only=True).to_frame(name="count").T

,n_erps,n_egas,n_others,total
count,115,0,0,115


In [ ]:
results_df.total.value_counts().to_frame(name="count").T

total,1,0
count,115,2


In [ ]:
results_df[results_df["total"] == 0]

,series,erps,egas,others,n_erps,n_egas,n_others,total
44,E-MTAB-6505,,,,0,0,0,0
46,E-MTAB-9216,,,,0,0,0,0


For `E-MTAB-6505` there is no matching ENA accession, but I see that `sdrf` file contains `biosample` identifiers. Let's check if I can find those samples in ENA

In [ ]:
read_enaruns(series="SAMEA5053920")

[{'run_accession': 'ERR2861957',
  'study_accession': 'PRJEB29431',
  'experiment_accession': 'ERX2868195'}]

In [ ]:
fields = "study_accession,secondary_study_accession,sample_accession,secondary_sample_accession,experiment_accession,experiment_alias,run_accession,run_alias,tax_id,scientific_name,fastq_ftp,submitted_ftp,sra_ftp"
read_enaruns(series="SAMEA5053921", fields=fields)

[{'run_accession': 'ERR2861958',
  'study_accession': 'PRJEB29431',
  'secondary_study_accession': 'ERP111731',
  'sample_accession': 'SAMEA5053921',
  'secondary_sample_accession': 'ERS2865099',
  'experiment_accession': 'ERX2868196',
  'experiment_alias': 'E-MTAB-6505:Day2_r2_p',
  'run_alias': 'E-MTAB-6505:Day2_r2',
  'tax_id': '9606',
  'scientific_name': 'Homo sapiens',
  'fastq_ftp': '',
  'submitted_ftp': 'ftp.sra.ebi.ac.uk/vol1/run/ERR286/ERR2861958/Day2_r2.bam;ftp.sra.ebi.ac.uk/vol1/run/ERR286/ERR2861958/Day2_r2.bam.bai',
  'sra_ftp': ''}]

In [ ]:
response = httpx.get("https://ftp.ebi.ac.uk/biostudies/fire/E-MTAB-/505/E-MTAB-6505/Files/E-MTAB-6505.sdrf.txt")
sdrf = pd.read_csv(io.StringIO(response.text), sep="\t")
sdrf

,Source Name,Comment[ENA_SAMPLE],Comment[BioSD_SAMPLE],Characteristics[organism],Characteristics[developmental stage],Characteristics[organism part],Characteristics[cell type],Characteristics[genotype],Characteristics[phenotype],Characteristics[growth condition],...,Comment[cell barcode read].1,Comment[cell barcode offset].1,Comment[cell barcode size].1,Comment[cDNA read].1,Comment[cDNA read offset].1,Comment[cDNA read size].1,Comment[read1 file].1,Comment[read2 file].1,Factor Value[time],Unit[time unit]
0,Day2_r1,ERS2865098,SAMEA5053920,Homo sapiens,neonate,umbilical cord blood,leukocyte,CD19.28z expression,CD45+; CD19.28z+,xenograft from HuSGM3 mouse infused with CD19....,...,read1,0,16,read2,0,96,Day2_r1_R1.fastq.gz,Day2_r1_R2.fastq.gz,2,day
1,Day2_r2,ERS2865099,SAMEA5053921,Homo sapiens,neonate,umbilical cord blood,leukocyte,CD19.28z expression,CD45+; CD19.28z+,xenograft from HuSGM3 mouse infused with CD19....,...,read1,0,16,read2,0,96,Day2_r2_R1.fastq.gz,Day2_r2_R2.fastq.gz,2,day
2,Day7_r1,ERS2865100,SAMEA5053922,Homo sapiens,neonate,umbilical cord blood,leukocyte,CD19.28z expression,CD45+; CD19.28z+,xenograft from HuSGM3 mouse infused with CD19....,...,read1,0,16,read2,0,96,Day7_r1_R1.fastq.gz,Day7_r1_R2.fastq.gz,7,day
3,Day7_r2,ERS2865101,SAMEA5053923,Homo sapiens,neonate,umbilical cord blood,leukocyte,CD19.28z expression,CD45+; CD19.28z+,xenograft from HuSGM3 mouse infused with CD19....,...,read1,0,16,read2,0,96,Day7_r2_R1.fastq.gz,Day7_r2_R2.fastq.gz,7,day


In [ ]:
sdrf.columns.tolist()

['Source Name',
 'Comment[ENA_SAMPLE]',
 'Comment[BioSD_SAMPLE]',
 'Characteristics[organism]',
 'Characteristics[developmental stage]',
 'Characteristics[organism part] ',
 'Characteristics[cell type] ',
 'Characteristics[genotype] ',
 'Characteristics[phenotype] ',
 'Characteristics[growth condition] ',
 'Characteristics[disease] ',
 'Material Type',
 'Comment[single cell isolation]',
 'Comment[single cell library construction]',
 'Comment[input molecule]',
 'Comment[primer]',
 'Comment[end bias]',
 'Description',
 'Protocol REF',
 'Protocol REF.1',
 'Extract Name',
 'Comment[LIBRARY_LAYOUT]',
 'Comment[LIBRARY_SELECTION]',
 'Comment[LIBRARY_SOURCE]',
 'Comment[LIBRARY_STRATEGY]',
 'Comment[LIBRARY_STRAND]',
 'Comment[NOMINAL_LENGTH]',
 'Comment[NOMINAL_SDEV]',
 'Protocol REF.2',
 'Performer',
 'Assay Name',
 'Comment[single cell isolation].1',
 'Comment[library construction]',
 'Comment[input molecule].1',
 'Comment[primer].1',
 'Comment[end bias].1',
 'Comment[umi barcode read]',
 